El servicio de venta de autos usados Rusty Bargain está desarrollando una aplicación para atraer nuevos clientes. Gracias a esa app, puedes averiguar rápidamente el valor de mercado de tu coche. Tienes acceso al historial: especificaciones técnicas, versiones de equipamiento y precios. Tienes que crear un modelo que determine el valor de mercado.
A Rusty Bargain le interesa:
- la calidad de la predicción;
- la velocidad de la predicción;
- el tiempo requerido para el entrenamiento

## Importación de las librerías

In [1]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

import lightgbm as lgb

## Preparación de datos

In [2]:
data = pd.read_csv('/datasets/car_data.csv')

## Revision de los datos

In [3]:
data.head()

,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
0,24/03/2016 11:52,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN,24/03/2016 00:00,0,70435,07/04/2016 03:16
1,24/03/2016 10:58,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes,24/03/2016 00:00,0,66954,07/04/2016 01:46
2,14/03/2016 12:52,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN,14/03/2016 00:00,0,90480,05/04/2016 12:47
3,17/03/2016 16:54,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no,17/03/2016 00:00,0,91074,17/03/2016 17:40
4,31/03/2016 17:25,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no,31/03/2016 00:00,0,60437,06/04/2016 10:17


In [4]:
data.shape
data.nunique().sort_values(ascending=False)

LastSeen             18592
DateCrawled          15470
PostalCode            8143
Price                 3731
Power                  712
Model                  250
RegistrationYear       151
DateCreated            109
Brand                   40
Mileage                 13
RegistrationMonth       13
VehicleType              8
FuelType                 7
Gearbox                  2
NotRepaired              2
NumberOfPictures         1
dtype: int64

## Limpieza de los datos

In [8]:
data = data.drop(['DateCrawled', 'DateCreated', 'LastSeen', 'PostalCode', 'NumberOfPictures'], axis=1, errors='ignore')
data = data.dropna()
data = data[data['Price'] > 0]

## Entrenamiento del modelo 

<b>Separacion de los datos.

In [9]:
features = data.drop('Price', axis = 1)
target = data['Price']

categorical_cols = features.select_dtypes(include='object').columns

features_ohe = pd.get_dummies(features, columns=categorical_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(features_ohe, target, test_size=0.2, random_state=12345)

<b>Entrenamiento del modelo 1 "Regresion lineal"

In [10]:
start = time.time()

lr = LinearRegression()
lr.fit(X_train, y_train)

train_time_lr = time.time() - start

start = time.time()
pred_lr = lr.predict(X_test)
predict_time_lr = time.time() - start

rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))

print("Linear Regression RMSE:", rmse_lr)
print("Train time:", train_time_lr)
print("Predict time:", predict_time_lr)

Linear Regression RMSE: 2674.7120112223492
Train time: 5.9769814014434814
Predict time: 0.10175800323486328


<b>Entrenamiento del segundo modelo "Arbol de desiciones"

In [12]:
start = time.time()

dt = DecisionTreeRegressor(
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=12345
)

dt.fit(X_train, y_train)
train_time_dt = time.time() - start

start = time.time()
pred_dt = dt.predict(X_test)
predict_time_dt = time.time() - start

rmse_dt = np.sqrt(mean_squared_error(y_test, pred_dt))

print("Decision Tree RMSE:", rmse_dt)
print("Train time:", train_time_dt)
print("Predict time:", predict_time_dt)

Decision Tree RMSE: 1798.194226887116
Train time: 3.2327709197998047
Predict time: 0.06797146797180176


<b>Entrenamiento del tercer modelo "Random Forest"

In [13]:
start = time.time()

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=12345,
    n_jobs=-1
)

rf.fit(X_train, y_train)
train_time_rf = time.time() - start

start = time.time()
pred_rf = rf.predict(X_test)
predict_time_rf = time.time() - start

rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))

print("Random Forest RMSE:", rmse_rf)
print("Train time:", train_time_rf)
print("Predict time:", predict_time_rf)

Random Forest RMSE: 1590.1961118089866
Train time: 100.60568499565125
Predict time: 0.5216901302337646


<b>Entrenamiento del cuarto modelo "LightGBM"

In [15]:
start = time.time()

lgb_model = lgb.LGBMRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=15,
    random_state=12345,
    n_jobs=-1
)

lgb_model.fit(X_train, y_train)
train_time_lgb = time.time() - start

start = time.time()
pred_lgb = lgb_model.predict(X_test)
predict_time_lgb = time.time() - start

rmse_lgb = np.sqrt(mean_squared_error(y_test, pred_lgb))

print("LightGBM RMSE:", rmse_lgb)
print("Train time:", train_time_lgb)
print("Predict time:", predict_time_lgb)

LightGBM RMSE: 1598.365922829213
Train time: 3.5022530555725098
Predict time: 0.3275268077850342


## Análisis de los modelos

<b>Comparacion final

In [16]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest', 'LightGBM'],
    'RMSE': [rmse_lr, rmse_dt, rmse_rf, rmse_lgb],
    'Train Time (s)': [train_time_lr, train_time_dt, train_time_rf, train_time_lgb],
    'Predict Time (s)': [predict_time_lr, predict_time_dt, predict_time_rf, predict_time_lgb]
})

results.sort_values('RMSE')

,Model,RMSE,Train Time (s),Predict Time (s)
2,Random Forest,1590.196112,100.605685,0.521690
3,LightGBM,1598.365923,3.502253,0.327527
1,Decision Tree,1798.194227,3.232771,0.067971
0,Linear Regression,2674.712011,5.976981,0.101758
